# HED Score — Interactive Demo

This notebook demonstrates the **Hiremath Early Detection (HED) Score**,
a measure-theoretic metric for evaluating early detection in time-series tasks.

**Reference**: Hiremath, P. S. (2026). *The Hiremath Early Detection (HED) Score.*  
arXiv:2604.04993 [stat.ML]

---
## 1. Install & import

In [ ]:
# Install if running in Colab
# !pip install hed-score matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt

from hed import hed_score
from hed.metrics import auc_score, hed_far_curve
from hed.utils import make_step_detector, make_ramp_detector, make_delayed_detector

print('HED package loaded successfully!')

---
## 2. Quick start — one function call

In [ ]:
T      = 200      # total time steps
t_star = 100      # regime shift at t=100
lam    = 0.1      # Hiremath Decay Constant

# A perfect detector: immediately outputs P=1 after the shift
P_perfect = np.zeros(T)
P_perfect[t_star:] = 1.0

score = hed_score(P_perfect, t_star, lam=lam)
print(f'HED (perfect detector) = {score:.4f}   [should be 1.0]')

---
## 3. The key insight: same AUC, different HED

In [ ]:
rng = np.random.default_rng(42)

P_fast    = make_step_detector(T, t_star, noise=0.04, rng=rng)
P_ramp    = make_ramp_detector(T, t_star, noise=0.04, rng=rng)
P_delayed = make_delayed_detector(T, t_star, delay=40, noise=0.04, rng=rng)

labels = np.zeros(T, dtype=int)
labels[t_star:] = 1

detectors = {
    'Fast (step)': P_fast,
    'Ramp (slow)': P_ramp,
    'Delayed':     P_delayed,
}

print(f'  {"Detector":<20} {"AUC":>8} {"HED":>10}')
print('  ' + '-' * 42)
for name, P in detectors.items():
    hed = hed_score(P, t_star, lam=lam)
    auc = auc_score(P, labels)
    print(f'  {name:<20} {auc:>8.4f} {hed:>10.4f}')

---
## 4. Visualise probability streams

In [ ]:
t = np.arange(T)
colors = ['#1565C0', '#E65100', '#AD1457']

fig, ax = plt.subplots(figsize=(10, 4))
for (name, P), color in zip(detectors.items(), colors):
    hed = hed_score(P, t_star, lam=lam)
    auc = auc_score(P, labels)
    ax.plot(t, P, color=color, lw=1.8, alpha=0.85,
            label=f'{name} | AUC={auc:.3f}, HED={hed:.3f}')

ax.axvline(t_star, color='black', ls='--', lw=1.5, label=f't* = {t_star}')
ax.set_xlabel('Time step'); ax.set_ylabel('P(anomalous)')
ax.set_title('Same AUC, different HED'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5. Effect of the decay constant λ

In [ ]:
lambdas = [0.01, 0.05, 0.1, 0.2, 0.5]

print('  λ        Fast HED    Ramp HED    Delayed HED')
print('  ' + '-' * 50)
for l in lambdas:
    hf = hed_score(P_fast, t_star, lam=l)
    hr = hed_score(P_ramp, t_star, lam=l)
    hd = hed_score(P_delayed, t_star, lam=l)
    print(f'  {l:<8}  {hf:>8.4f}    {hr:>8.4f}    {hd:>11.4f}')

---
## 6. FAR–HED curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for (name, P), color in zip(detectors.items(), colors):
    far, hed = hed_far_curve(P, t_star, lam=lam, n_thresholds=100)
    order = np.argsort(far)
    ax.plot(far[order], hed[order], color=color, lw=2, label=name)

ax.scatter([0], [1], color='gold', s=100, zorder=5, edgecolor='k', label='Ideal')
ax.set_xlabel('False Alarm Rate'); ax.set_ylabel(f'HED (λ={lam})')
ax.set_title('FAR–HED Curve  (upper-left = better)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. Hiremath Standard Table

| Domain | Recommended λ | Rationale |
|---|---|---|
| Cyber-physical security | 0.3 | Every second of early warning is critical |
| Epidemiological monitoring | 0.05 | Slower-moving outbreaks, longer horizon |
| Algorithmic surveillance | 0.2 | Market movements can be rapid |
| General / exploratory | 0.1 | Balanced default |